In [1]:
import torch
import torch.nn as nn
torch.set_default_dtype(torch.float64)
from collections import OrderedDict
import math
import numpy as np
import sympy as sp
import seaborn as sns
from scipy.interpolate import griddata
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import make_axes_locatable
import pandas as pd
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import cm
from scipy.stats import qmc
import scipy.io

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
class Sin(torch.nn.Module):
    """sin 激活: y = sin(omega * x)"""
    def __init__(self, omega=1.0):
        super(Sin, self).__init__()
        self.omega = omega

    def forward(self, x):
        return torch.sin(self.omega * x)


class DNN1(torch.nn.Module):
    def __init__(self, layers, lb, ub):
        super(DNN1, self).__init__()
        
        self.depth = len(layers) - 1
        
        # 设置激活函数 (消融: tanh -> sin, 参考 PgMSNN; omega=1, SIREN 风格可试 10~30)
        self.activation1 = Sin
        
        layer_list = list()
        for i in range(self.depth - 1): 
            layer_list.append(
                ('layer_%d' % i, torch.nn.Linear(layers[i], layers[i+1]))
            )
            layer_list.append(('activation_%d' % i, self.activation1()))
            
        layer_list.append(
            ('layer_%d' % (self.depth - 1), torch.nn.Linear(layers[-2], layers[-1]))
        )
        
        layerDict = OrderedDict(layer_list)
        
        # 部署层
        self.layers = torch.nn.Sequential(layerDict)

        self.lb = lb
        self.ub = ub
        
    def forward(self, X):
        out = 2 * (X - self.lb) / (self.ub - self.lb) - 1
        out = self.layers(out)
        return out
    
    def weights_init(m):
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            nn.init.xavier_normal_(m.weight)
            nn.init.constant_(m.bias, 0.0)


In [4]:
# the physics-guided neural network
class PINN(nn.Module):
    def __init__(self, X_ub, X_lb, X_ic, X_res, q_ic, r_ic, layers, lb, ub):
        super(PINN, self).__init__()
        # boundary conditions
        
        self.z_ub = torch.tensor(X_ub[:, 0:1], requires_grad=True).double().to(device)
        self.t_ub = torch.tensor(X_ub[:, 1:2], requires_grad=True).double().to(device)

        self.z_lb = torch.tensor(X_lb[:, 0:1], requires_grad=True).double().to(device)
        self.t_lb = torch.tensor(X_lb[:, 1:2], requires_grad=True).double().to(device)

        self.z_ic = torch.tensor(X_ic[:, 0:1], requires_grad=True).double().to(device)
        self.t_ic = torch.tensor(X_ic[:, 1:2], requires_grad=True).double().to(device)
        self.q_ic = torch.tensor(q_ic).double().to(device)
        self.r_ic = torch.tensor(r_ic).double().to(device)

        self.z_res = torch.tensor(X_res[:, 0:1], requires_grad=True).double().to(device)
        self.t_res = torch.tensor(X_res[:, 1:2], requires_grad=True).double().to(device)

        self.lb = torch.tensor(lb).double().to(device)  # 归一化下界
        self.ub = torch.tensor(ub).double().to(device)  # 归一

        self.layers = layers
        
        self.net = DNN1(layers,self.lb, self.ub).to(device)

        self.iter = 1

        self.optimizer1 = torch.optim.Adam(self.net.parameters(),lr=1e-3)

        self.optimizer2 = torch.optim.LBFGS(
            self.net.parameters(), 
            lr=1.0, #步长缩放因子，用于控制拟牛顿算法中搜索的步长
            max_iter=30000, #最大迭代次数
            max_eval=30000, #函数和梯度最大评估次数
            history_size=50, #存储和更新历史信息的缓存大小
            tolerance_grad=1e-7, #表示梯度的容差，即梯度的变化小于该容差时，算法将终止
            tolerance_change=1.0 * np.finfo(float).eps, #表示优化变化的容差，即参数变化量的绝对值小于该容差时，算法将终止
            line_search_fn="strong_wolfe" #can be "strong_wolfe"
        )
        
        self.loss_history = {
            'total': [],
            'residual': [],
            'ic': [],
            'b': []
        }

    def net_h(self, z, t):  
        H = self.net(torch.cat([z, t], dim=1))
        q = H[:, 0:1]
        r = H[:, 1:2]
        return q, r
        
    def gradient(self, y, x):
        return torch.autograd.grad(y, x, grad_outputs=torch.ones_like(y), retain_graph=True, create_graph=True)[0]
    
    def net_b(self, z, t):

        q, r = self.net_h(z, t)
        q_z = self.gradient(q, z)
        r_z = self.gradient(r, z)

        return q, r, q_z, r_z                   
    
    def net_f(self, z, t):

        q, r = self.net_h(z, t)
        q_t = self.gradient(q, t)
        q_z = self.gradient(q, z)

        r_t = self.gradient(r, t)
        r_z = self.gradient(r, z)
        r_zz = self.gradient(r_z, z)
        r_zzz = self.gradient(r_zz, z)

        f1 = r_t - 4 * q_z - 6 * r * r_z
        # 8.29: 非线性系数改回 (2, 4), 与 ADM.py 递推和谱方法参考数据一致
        f2 = q_t - r_zzz - 2 * r * q_z - 4 * q * r_z

        return f1, f2

    def loss_func(self):

        self.optimizer1.zero_grad()
        self.optimizer2.zero_grad()

        q_ic_pred, r_ic_pred = self.net_h(self.z_ic, self.t_ic)

        q_ub_pred, r_ub_pred, q_z_ub_pred, r_z_ub_pred = self.net_b(self.z_ub, self.t_ub)

        q_lb_pred, r_lb_pred, q_z_lb_pred, r_z_lb_pred = self.net_b(self.z_lb, self.t_lb)

        f1_res_pred, f2_res_pred = self.net_f(self.z_res, self.t_res)

        loss_res1 = torch.mean(f1_res_pred**2)
        loss_res2 = torch.mean(f2_res_pred**2)
        loss_icq = torch.mean((q_ic_pred - self.q_ic)**2)
        loss_icr = torch.mean((r_ic_pred - self.r_ic)**2)

        # 周期边界: 两端的值与导数对应相等 (与谱方法格式一致)
        loss_b = (torch.mean((r_ub_pred - r_lb_pred)**2)
                + torch.mean((q_ub_pred - q_lb_pred)**2)
                + torch.mean((r_z_ub_pred - r_z_lb_pred)**2)
                + torch.mean((q_z_ub_pred - q_z_lb_pred)**2))

        loss_ic = loss_icq + loss_icr
        loss_res = loss_res1 + loss_res2

        loss = loss_res + loss_ic + loss_b
        
        loss.backward()
        self.iter += 1
        if self.iter % 100 == 0:
            self.loss_history['total'].append(loss.item())
            self.loss_history['residual'].append(loss_res.item())
            self.loss_history['ic'].append(loss_ic.item())
            self.loss_history['b'].append(loss_b.item())
            print(
                'Iter %d, Loss: %.5e, Loss_res: %.5e, Loss_ic: %.5e, Loss_b: %.5e' % 
                (   self.iter, 
                    loss.item(), 
                    loss_res.item(), 
                    loss_ic.item(),
                    loss_b.item()
                )
            )

        return loss
    
    def train(self):
        self.net.train()
        print("采用Adam优化器")
        for i in range(10000):
            self.optimizer1.step(self.loss_func)
        # 然后运行lbfgs优化器
        print("采用L-BFGS优化器")
        self.optimizer2.step(self.loss_func) 
            
    def predict(self, X):
        
        z = torch.tensor(X[:, 0:1], requires_grad=True).double().to(device)
        t = torch.tensor(X[:, 1:2], requires_grad=True).double().to(device)

        self.net.eval()
        q, r = self.net_h(z, t)
        q = q.detach().cpu().numpy()
        r = r.detach().cpu().numpy()

        return q, r

In [ ]:
N_res = 10000
N_b = 400
N_ic = 1000

x = np.linspace(-15, 15, 601)      # 8.29: 计算域扩到 [-15,15]
t = np.linspace(0, 0.3, 201)       

x_mesh, t_mesh = np.meshgrid(x, t)

X = np.hstack((x_mesh.flatten()[:,None], t_mesh.flatten()[:,None]))

lb = X.min(0) #下界x'QxQ
ub = X.max(0)  #上界 

def lhs(n, size):
    """生成n维拉丁超立方采样，返回size个样本点"""
    sampler = qmc.LatinHypercube(d=n)
    return sampler.random(n=size)

# 使用简洁形式进行采样（与你要求的形式一致）
X_res = lb + (ub - lb) * lhs(2, N_res)  # 2维残差点采样

# 边界点和初始条件点也采用类似风格的简洁写法
X_ub = np.hstack((ub[0] * np.ones((N_b, 1)), 
                  lb[1] + (ub[1] - lb[1]) * lhs(1, N_b)))  # 上边界点

X_lb = np.hstack((lb[0] * np.ones((N_b, 1)), 
                  lb[1] + (ub[1] - lb[1]) * lhs(1, N_b)))  # 下边界点

X_ic = np.hstack((lb[0] + (ub[0] - lb[0]) * lhs(1, N_ic), 
                  lb[1] * np.ones((N_ic, 1))))  # 初始条件点

In [6]:
def funic(X):
    x = X[:, 0:1]
    # 8.29: 初始条件改为摊平高斯包 + q0 = 0
    # (与谱方法参考数据一致; 纯高斯与带限版差异 <2e-4, PINN 中直接用纯高斯即可)
    r = 0.3 * np.exp(-0.25 * x**2)
    q = np.zeros_like(x)
    return r, q

r_ic, q_ic = funic(X_ic)

In [7]:
layers = [2] + 6 * [80] + [2]

model = PINN(X_ub, X_lb, X_ic, X_res, q_ic, r_ic, layers, lb, ub)

In [8]:
def count_parameters(model):
    """计算模型中可训练参数的总个数"""
    total_params = 0
    for param in model.parameters():
        # 累加每个参数张量的元素数量
        total_params += param.numel()
    return total_params

param_count = count_parameters(model)
print(f"模型可训练参数总个数: {param_count:,}")  # 使用逗号作为千位分隔符

模型可训练参数总个数: 32,802


In [9]:
model.train()

采用Adam优化器
Iter 100, Loss: 5.67155e-03, Loss_res: 9.59552e-05, Loss_ic: 5.27950e-03, Loss_b: 2.96089e-04
Iter 200, Loss: 5.28124e-03, Loss_res: 4.05069e-05, Loss_ic: 4.88727e-03, Loss_b: 3.53468e-04
Iter 300, Loss: 4.99132e-03, Loss_res: 4.05919e-05, Loss_ic: 4.69126e-03, Loss_b: 2.59464e-04
Iter 400, Loss: 3.98043e-03, Loss_res: 3.46409e-05, Loss_ic: 3.91475e-03, Loss_b: 3.10376e-05
Iter 500, Loss: 3.59195e-03, Loss_res: 5.23893e-05, Loss_ic: 3.42311e-03, Loss_b: 1.16454e-04
Iter 600, Loss: 5.23768e-03, Loss_res: 5.49351e-05, Loss_ic: 3.43559e-03, Loss_b: 1.74715e-03
Iter 700, Loss: 2.22557e-03, Loss_res: 1.01602e-04, Loss_ic: 2.11632e-03, Loss_b: 7.64951e-06
Iter 800, Loss: 2.12164e-03, Loss_res: 1.26990e-04, Loss_ic: 1.98867e-03, Loss_b: 5.98359e-06
Iter 900, Loss: 1.07270e-03, Loss_res: 1.06721e-04, Loss_ic: 9.57532e-04, Loss_b: 8.45194e-06
Iter 1000, Loss: 6.20744e-04, Loss_res: 2.76562e-04, Loss_ic: 3.42429e-04, Loss_b: 1.75255e-06
Iter 1100, Loss: 5.28295e-04, Loss_res: 2.52734e-

In [10]:
q_pred, r_pred = model.predict(X)

In [11]:
H_pred = griddata(X, q_pred.flatten(), (x_mesh,t_mesh), method='cubic')
H_True = griddata(X, r_pred.flatten(), (x_mesh,t_mesh), method='cubic')

In [ ]:
def plot_all_3d(T, X, H_True, H_pred, smooth_factor=1.0, dpi=150):
    """
    绘制对比3D图像（真实值、预测值和差值），并优化显示效果
    
    参数:
    - smooth_factor: 控制插值平滑程度的因子
    - dpi: 图像分辨率
    """
    # 创建更高分辨率的图形
    fig = plt.figure(figsize=(21, 5), dpi=dpi)
    
    # 定义共享的颜色映射和视角
    cmap = cm.get_cmap('rainbow')  # 
    #cmap = cm.get_cmap('viridis')  # 使用更现代的颜色映射
    elev, azim = 30, 45  # 定义共享的视角
    
    # 创建更平滑的网格数据（可选）
    if smooth_factor > 1:
        from scipy.ndimage import gaussian_filter
        H_True_smooth = gaussian_filter(H_True, sigma=smooth_factor)
        H_pred_smooth = gaussian_filter(H_pred, sigma=smooth_factor)
    else:
        H_True_smooth = H_True
        H_pred_smooth = H_pred
    
    # 计算误差
    error = np.abs(H_True - H_pred)
    
    # 定义绘图函数，减少代码重复
    def plot_surface(ax, X, Y, Z, title, cmap=cmap):
        surf = ax.plot_surface(X, Y, Z, cmap=cmap, 
                              linewidth=0, antialiased=True,
                              rstride=1, cstride=1,
                              alpha=0.95)
        ax.set_xlabel('$Tv$')
        ax.set_ylabel('$Z$')
        ax.set_zlabel('h')
        ax.set_title(title)
        ax.view_init(elev=elev, azim=azim)
        fig.colorbar(surf, ax=ax, shrink=0.5, aspect=10, pad=0.05)
        # 添加网格线增强深度感
        ax.grid(True, linestyle='-', alpha=0.3)
        return surf
    
    # 真实值3D图
    ax0 = fig.add_subplot(131, projection='3d')
    plot_surface(ax0, T, X, H_True_smooth, 'Real')
    
    # 预测值3D图
    ax1 = fig.add_subplot(132, projection='3d')
    plot_surface(ax1, T, X, H_pred_smooth, 'Prediction')
    
    # 差距3D图
    ax2 = fig.add_subplot(133, projection='3d')
    # 使用专门的误差颜色映射
    #error_cmap = cm.get_cmap('Reds')
    plot_surface(ax2, T, X, error, 'Gap')
    
    # 调整布局，确保有足够的空间显示标签和颜色条
    plt.tight_layout(pad=3.0)
    plt.show()

# 示例调用
plot_all_3d(x_mesh, t_mesh, H_True, H_pred, smooth_factor=1.2, dpi=200)

In [13]:
data = {
    'x': x_mesh.flatten()[:, None],
    't': t_mesh.flatten()[:, None],
    'q_pred': q_pred.flatten()[:, None],
    'r_pred': r_pred.flatten()[:, None]
}

scipy.io.savemat('/kaggle/working/1all_data.mat', data)

In [14]:
loss_total = model.loss_history['total']
loss_residual = model.loss_history['residual']
loss_ic = model.loss_history['ic']
loss_b = model.loss_history['b']

In [15]:
num_iterations = len(loss_total)  # 损失记录的总数量（即总迭代次数）
iter_steps = list(range(100, 100 + num_iterations * 100, 100))
# 3. 将数据转换为竖排（列向量）：通过reshape(-1, 1)实现，形状变为(n, 1)
iter_steps_col = np.array(iter_steps).reshape(-1, 1)       # 竖排迭代步数
loss_total_col = np.array(loss_total).reshape(-1, 1)       # 竖排总损失
loss_residual_col = np.array(loss_residual).reshape(-1, 1) # 竖排残差损失
loss_ic_col = np.array(loss_ic).reshape(-1, 1)             # 竖排初始条件损失
loss_b_col = np.array(loss_b).reshape(-1, 1)               # 竖排边界条件损失

# 4. 整理并保存为mat文件（此时所有数据均为列向量）
data_loss = {
    'iter_steps': iter_steps_col,
    'loss_total': loss_total_col,
    'loss_residual': loss_residual_col,
    'loss_ic': loss_ic_col,
    'loss_b': loss_b_col
}

scipy.io.savemat('/kaggle/working/1loss_history.mat', data_loss)